### Overview

Zo všetkých posudzovaných variantov sa ako najlepší ukázal model logistickej regresie s normalizáciou atribútov a výberom hyperparametrov, ktorý zabezpečil najvyššiu hodnotu F1 a najvyváženejší pomer medzi presnosťou a recallom.

Naopak najhoršie výsledky dosiahla model s vyvažovaním tried (class_weight=„balanced“), ktorý napriek vysokému recall mal veľmi nízku presnosť, čo viedlo k výraznému zníženiu F1-miery.


In [9]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,confusion_matrix)
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV

In [10]:
TW_500= pd.read_csv(
    "../classification/Twitter/Relative_labeling/sigma=500/Twitter-Relative-Sigma-500.data",
    sep=",",
    header=None
)

TW_1000= pd.read_csv(
    "../classification/Twitter/Relative_labeling/sigma=1000/Twitter-Relative-Sigma-1000.data",
    sep=",",
    header=None
)
TW_1500= pd.read_csv(
    "../classification/Twitter/Relative_labeling/sigma=1500/Twitter-Relative-Sigma-1500.data",
    sep=",",
    header=None
)
groups = ["NCD", 'AI', 'AS(NA)', 'BL',
         'NAC', 'AS(NAC)', 'CS', 'AT', 'NA','ADL', 'NAD']

columns = []
for group in groups:
    for t in range(7):
        columns.append(f"{group}_{t}")

columns.append("label") 

TW_500.columns = columns
TW_1000.columns = columns
TW_1500.columns = columns


### 500

### 1.Baseline Logistic Regression without scaling ###

In [3]:
X = TW_500.drop("label", axis=1)
y = TW_500["label"]


X_train, X_test, y_train, y_test = train_test_split( X, y,test_size=0.2,random_state=42,stratify=y)
model = LogisticRegression(max_iter=1000).fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

cm = confusion_matrix(y_test, y_pred)
print('1.Baseline Logistic Regression without scaling')
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)
print("ROC-AUC:", roc_auc)

print("\nConfusion Matrix:")
print(cm)

1.Baseline Logistic Regression without scaling
Accuracy: 0.979354701158411
Precision: 0.6948228882833788
Recall: 0.35220994475138123
F1-score: 0.4674610449129239
ROC-AUC: 0.90022760988164

Confusion Matrix:
[[27306   112]
 [  469   255]]


### 2. Baseline Logistic Regression with scaling

In [4]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model_scaled = LogisticRegression(max_iter=1000).fit(X_train_scaled, y_train)
y_pred_scaled = model_scaled.predict(X_test_scaled)
y_prob_scaled = model_scaled.predict_proba(X_test_scaled)[:,1]

accuracy_scaled = accuracy_score(y_test, y_pred_scaled)
precision_scaled = precision_score(y_test, y_pred_scaled)
recall_scaled = recall_score(y_test, y_pred_scaled)
f1_scaled = f1_score(y_test, y_pred_scaled)
roc_auc_scaled = roc_auc_score(y_test, y_prob_scaled)

cm_scaled = confusion_matrix(y_test, y_pred_scaled)

print('2.Baseline Logistic Regression with StandardScaler')
print("Accuracy:", accuracy_scaled)
print("Precision:", precision_scaled)
print("Recall:", recall_scaled)
print("F1-score:", f1_scaled)
print("ROC-AUC:", roc_auc_scaled)

print("\nConfusion Matrix:")
print(cm_scaled)

2.Baseline Logistic Regression with StandardScaler
Accuracy: 0.9807405301684315
Precision: 0.7407407407407407
Recall: 0.3867403314917127
F1-score: 0.5081669691470054
ROC-AUC: 0.95100019989288

Confusion Matrix:
[[27320    98]
 [  444   280]]


### 3. Balanced Logistic Regression with scaling

In [5]:
model_balanced = LogisticRegression(max_iter=1000,class_weight='balanced')
model_balanced.fit(X_train_scaled, y_train)

y_pred_balanced = model_balanced.predict(X_test_scaled)
y_prob_balanced = model_balanced.predict_proba(X_test_scaled)[:,1]

accuracy_balanced = accuracy_score(y_test, y_pred_balanced)
precision_balanced = precision_score(y_test, y_pred_balanced)
recall_balanced = recall_score(y_test, y_pred_balanced)
f1_balanced = f1_score(y_test, y_pred_balanced)
roc_auc_balanced = roc_auc_score(y_test, y_prob_balanced)

cm_balanced = confusion_matrix(y_test, y_pred_balanced)

print('3. Balanced Logistic Regression with scaling')
print("Accuracy:", accuracy_balanced)
print("Precision:", precision_balanced)
print("Recall:", recall_balanced)
print("F1-score:", f1_balanced)
print("ROC-AUC:", roc_auc_balanced)

print("\nConfusion Matrix:")
print(cm_balanced)

3. Balanced Logistic Regression with scaling
Accuracy: 0.9216828938952455
Precision: 0.22972972972972974
Recall: 0.8687845303867403
F1-score: 0.3633737723859041
ROC-AUC: 0.9625319234168463

Confusion Matrix:
[[25309  2109]
 [   95   629]]


### 4. Logistic Regression with Stratified K-Fold using pipeline and scaling

In [6]:
pipeline = Pipeline([("scaler", StandardScaler()),("model", LogisticRegression(max_iter=1000))])

cv = StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

cv_results = cross_validate( pipeline,X,y, cv=cv,scoring=scoring)

print('Logistic Regression with Stratified K-Fold using pipeline and scaling')
print("Accuracy:", cv_results["test_accuracy"].mean())
print("Precision:", cv_results["test_precision"].mean())
print("Recall:", cv_results["test_recall"].mean())
print("F1-score:", cv_results["test_f1"].mean())
print("ROC-AUC:", cv_results["test_roc_auc"].mean())

Logistic Regression with Stratified K-Fold using pipeline and scaling
Accuracy: 0.9802355237678656
Precision: 0.7258292301680948
Recall: 0.3726519337016575
F1-score: 0.4922641039704315
ROC-AUC: 0.9531837324797479


### 5. Logistic Regression with Grid Search using pipeline and scaling 

In [14]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000
    ))
])

param_dist = {
    "model__C": [0.01, 0.1, 1, 10],
    "model__penalty": ["l1", "l2"],
    "model__solver": ["liblinear"]
}

search = RandomizedSearchCV(pipeline, param_distributions=param_dist,n_iter=5, cv=3, scoring="f1_weighted",n_jobs=-1,verbose=2,random_state=42)

search.fit(X, y)

best_model = search.best_estimator_

print("Best parameters:", search.best_params_)

y_pred = best_model.predict(X)
y_prob = best_model.predict_proba(X)[:, 1]

accuracy = accuracy_score(y, y_pred)
precision = precision_score(y, y_pred)
recall = recall_score(y, y_pred)
f1 = f1_score(y, y_pred)
roc_auc = roc_auc_score(y, y_prob)

cm = confusion_matrix(y, y_pred)

print("\n=== METRICS ===")
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)
print("ROC-AUC:", roc_auc)

print("\nConfusion Matrix:")
print(cm)


Fitting 3 folds for each of 5 candidates, totalling 15 fits
Best parameters: {'model__solver': 'liblinear', 'model__penalty': 'l2', 'model__C': 1}

=== METRICS ===
Accuracy: 0.9803847711911987
Precision: 0.7304394426580921
Recall: 0.3765193370165746
F1-score: 0.4969012030623405
ROC-AUC: 0.9555333857230721

Confusion Matrix:
[[136584    503]
 [  2257   1363]]


In [2]:
import pandas as pd

results = [
    {
        "Model": "LR baseline",
        "Accuracy": 0.97935,
        "Precision": 0.6948,
        "Recall": 0.3522,
        "F1": 0.4675,
        "ROC-AUC": 0.9002
    },
    {
        "Model": "LR + scaling",
        "Accuracy": 0.98074,
        "Precision": 0.7407,
        "Recall": 0.3867,
        "F1": 0.5082,
        "ROC-AUC": 0.9510
    },
    {
        "Model": "LR balanced",
        "Accuracy": 0.92168,
        "Precision": 0.2297,
        "Recall": 0.8688,
        "F1": 0.3634,
        "ROC-AUC": 0.9625
    },
    {
        "Model": "LR + Stratified CV",
        "Accuracy": 0.98024,
        "Precision": 0.7258,
        "Recall": 0.3727,
        "F1": 0.4923,
        "ROC-AUC": 0.9532
    },
    {
        "Model": "LR tuned",
        "Accuracy": 0.98038,
        "Precision": 0.7304,
        "Recall": 0.3765,
        "F1": 0.4969,
        "ROC-AUC": 0.9555
    }
]

df_results = pd.DataFrame(results)

df_results = df_results.sort_values(by="F1", ascending=False)

df_results

,Model,Accuracy,Precision,Recall,F1,ROC-AUC
1,LR + scaling,0.98074,0.7407,0.3867,0.5082,0.9510
4,LR tuned,0.98038,0.7304,0.3765,0.4969,0.9555
3,LR + Stratified CV,0.98024,0.7258,0.3727,0.4923,0.9532
0,LR baseline,0.97935,0.6948,0.3522,0.4675,0.9002
2,LR balanced,0.92168,0.2297,0.8688,0.3634,0.9625


Zo všetkých posudzovaných variantov sa ako najlepší ukázal model logistickej regresie s normalizáciou atribútov a výberom hyperparametrov, ktorý zabezpečil najvyššiu hodnotu F1 a najvyváženejší pomer medzi presnosťou a recallom.

Naopak najhoršie výsledky dosiahla model s vyvažovaním tried (class_weight=„balanced“), ktorý napriek vysokému recall mal veľmi nízku presnosť, čo viedlo k výraznému zníženiu F1-miery.


### 1000

### 1.Baseline Logistic Regression without scaling ###

In [16]:
X = TW_1000.drop("label", axis=1)
y = TW_1000["label"]


X_train, X_test, y_train, y_test = train_test_split( X, y,test_size=0.2,random_state=42,stratify=y)
model = LogisticRegression(max_iter=1000).fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

cm = confusion_matrix(y_test, y_pred)
print('1.Baseline Logistic Regression without scaling')
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)
print("ROC-AUC:", roc_auc)

print("\nConfusion Matrix:")
print(cm)

1.Baseline Logistic Regression without scaling
Accuracy: 0.9937460024163173
Precision: 0.7122302158273381
Recall: 0.42127659574468085
F1-score: 0.5294117647058824
ROC-AUC: 0.9313318019043495

Confusion Matrix:
[[27867    40]
 [  136    99]]


### 2. Baseline Logistic Regression with scaling

In [17]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model_scaled = LogisticRegression(max_iter=1000).fit(X_train_scaled, y_train)
y_pred_scaled = model_scaled.predict(X_test_scaled)
y_prob_scaled = model_scaled.predict_proba(X_test_scaled)[:,1]

accuracy_scaled = accuracy_score(y_test, y_pred_scaled)
precision_scaled = precision_score(y_test, y_pred_scaled)
recall_scaled = recall_score(y_test, y_pred_scaled)
f1_scaled = f1_score(y_test, y_pred_scaled)
roc_auc_scaled = roc_auc_score(y_test, y_prob_scaled)

cm_scaled = confusion_matrix(y_test, y_pred_scaled)

print('2.Baseline Logistic Regression with StandardScaler')
print("Accuracy:", accuracy_scaled)
print("Precision:", precision_scaled)
print("Recall:", recall_scaled)
print("F1-score:", f1_scaled)
print("ROC-AUC:", roc_auc_scaled)

print("\nConfusion Matrix:")
print(cm_scaled)

2.Baseline Logistic Regression with StandardScaler
Accuracy: 0.9943500817283775
Precision: 0.7533333333333333
Recall: 0.4808510638297872
F1-score: 0.587012987012987
ROC-AUC: 0.9647407917940212

Confusion Matrix:
[[27870    37]
 [  122   113]]


### 3. Balanced Logistic Regression with scaling

In [18]:
model_balanced = LogisticRegression(max_iter=1000,class_weight='balanced')
model_balanced.fit(X_train_scaled, y_train)

y_pred_balanced = model_balanced.predict(X_test_scaled)
y_prob_balanced = model_balanced.predict_proba(X_test_scaled)[:,1]

accuracy_balanced = accuracy_score(y_test, y_pred_balanced)
precision_balanced = precision_score(y_test, y_pred_balanced)
recall_balanced = recall_score(y_test, y_pred_balanced)
f1_balanced = f1_score(y_test, y_pred_balanced)
roc_auc_balanced = roc_auc_score(y_test, y_prob_balanced)

cm_balanced = confusion_matrix(y_test, y_pred_balanced)

print('3. Balanced Logistic Regression with scaling')
print("Accuracy:", accuracy_balanced)
print("Precision:", precision_balanced)
print("Recall:", recall_balanced)
print("F1-score:", f1_balanced)
print("ROC-AUC:", roc_auc_balanced)

print("\nConfusion Matrix:")
print(cm_balanced)

3. Balanced Logistic Regression with scaling
Accuracy: 0.9575723118470614
Precision: 0.15127272727272728
Recall: 0.8851063829787233
F1-score: 0.25838509316770186
ROC-AUC: 0.9715411903823412

Confusion Matrix:
[[26740  1167]
 [   27   208]]


### 4. Logistic Regression with Stratified K-Fold using pipeline and scaling

In [19]:
pipeline = Pipeline([("scaler", StandardScaler()),("model", LogisticRegression(max_iter=1000))])

cv = StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

cv_results = cross_validate( pipeline,X,y, cv=cv,scoring=scoring)

print('Logistic Regression with Stratified K-Fold using pipeline and scaling')
print("Accuracy:", cv_results["test_accuracy"].mean())
print("Precision:", cv_results["test_precision"].mean())
print("Recall:", cv_results["test_recall"].mean())
print("F1-score:", cv_results["test_f1"].mean())
print("ROC-AUC:", cv_results["test_roc_auc"].mean())

Logistic Regression with Stratified K-Fold using pipeline and scaling
Accuracy: 0.9941438542735789
Precision: 0.7368441522032775
Recall: 0.46636855391272986
F1-score: 0.570546798151416
ROC-AUC: 0.968037452067524


### 5. Logistic Regression with Grid Search using pipeline and scaling

In [21]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000
    ))
])

param_dist = {
    "model__C": [0.01, 0.1, 1, 10],
    "model__penalty": ["l1", "l2"],
    "model__solver": ["liblinear"]
}

search = RandomizedSearchCV( pipeline,param_distributions=param_dist,n_iter=5,cv=3,scoring="f1_weighted",n_jobs=-1,verbose=2,random_state=42)

search.fit(X, y)

best_model = search.best_estimator_

print("Best parameters:", search.best_params_)

y_pred = best_model.predict(X)
y_prob = best_model.predict_proba(X)[:, 1]

accuracy = accuracy_score(y, y_pred)
precision = precision_score(y, y_pred)
recall = recall_score(y, y_pred)
f1 = f1_score(y, y_pred)
roc_auc = roc_auc_score(y, y_prob)

cm = confusion_matrix(y, y_pred)

print("Random Search")
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)
print("ROC-AUC:", roc_auc)

print("\nConfusion Matrix:")
print(cm)


Fitting 3 folds for each of 5 candidates, totalling 15 fits
Best parameters: {'model__solver': 'liblinear', 'model__penalty': 'l2', 'model__C': 1}
Random Search
Accuracy: 0.9943570682339897
Precision: 0.7549933422103862
Recall: 0.4817332200509771
F1-score: 0.5881742738589212
ROC-AUC: 0.9723282087741948

Confusion Matrix:
[[139346    184]
 [   610    567]]


In [6]:
import pandas as pd

results_1000 = [
    {"Model": "LR baseline","Accuracy": 0.99375, "Precision": 0.7122, "Recall": 0.4213,"F1": 0.5294,"ROC-AUC": 0.9313},
    {
        "Model": "LR + scaling",
        "Accuracy": 0.99435,
        "Precision": 0.7533,
        "Recall": 0.4809,
        "F1": 0.5870,
        "ROC-AUC": 0.9647
    },
    {
        "Model": "LR balanced",
        "Accuracy": 0.95757,
        "Precision": 0.1513,
        "Recall": 0.8851,
        "F1": 0.2584,
        "ROC-AUC": 0.9715
    },
    {
        "Model": "LR + Stratified CV",
        "Accuracy": 0.99414,
        "Precision": 0.7368,
        "Recall": 0.4664,
        "F1": 0.5705,
        "ROC-AUC": 0.9680
    },
    {
        "Model": "LR tuned",
        "Accuracy": 0.99436,
        "Precision": 0.7550,
        "Recall": 0.4817,
        "F1": 0.5882,
        "ROC-AUC": 0.9723
    }
]

df_1000 = pd.DataFrame(results_1000)

df_1000 = df_1000.sort_values(by="F1", ascending=False)

df_1000

,Model,Accuracy,Precision,Recall,F1,ROC-AUC
4,LR tuned,0.99436,0.7550,0.4817,0.5882,0.9723
1,LR + scaling,0.99435,0.7533,0.4809,0.5870,0.9647
3,LR + Stratified CV,0.99414,0.7368,0.4664,0.5705,0.9680
0,LR baseline,0.99375,0.7122,0.4213,0.5294,0.9313
2,LR balanced,0.95757,0.1513,0.8851,0.2584,0.9715


Zo všetkých variantov sa opäť ako najlepší ukázal model logistickej regresie so škálovaním a výberom hyperparametrov.
Najhoršie výsledky dosiahol model s vyvažovaním tried, ktorý napriek vysokej hodnote recall mal veľmi nízku presnosť, čo negatívne ovplyvnilo celkovú kvalitu klasifikácie.


### 1500

### 1.Baseline Logistic Regression without scaling ###

In [22]:
X = TW_1500.drop("label", axis=1)
y = TW_1500["label"]


X_train, X_test, y_train, y_test = train_test_split( X, y,test_size=0.2,random_state=42,stratify=y)
model = LogisticRegression(max_iter=1000).fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

cm = confusion_matrix(y_test, y_pred)
print('1.Baseline Logistic Regression without scaling')
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)
print("ROC-AUC:", roc_auc)

print("\nConfusion Matrix:")
print(cm)

1.Baseline Logistic Regression without scaling
Accuracy: 0.9973349442114988
Precision: 0.6825396825396826
Recall: 0.4387755102040816
F1-score: 0.5341614906832298
ROC-AUC: 0.9615680461315891

Confusion Matrix:
[[28024    20]
 [   55    43]]


### 2. Baseline Logistic Regression with scaling

In [23]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model_scaled = LogisticRegression(max_iter=1000).fit(X_train_scaled, y_train)
y_pred_scaled = model_scaled.predict(X_test_scaled)
y_prob_scaled = model_scaled.predict_proba(X_test_scaled)[:,1]

accuracy_scaled = accuracy_score(y_test, y_pred_scaled)
precision_scaled = precision_score(y_test, y_pred_scaled)
recall_scaled = recall_score(y_test, y_pred_scaled)
f1_scaled = f1_score(y_test, y_pred_scaled)
roc_auc_scaled = roc_auc_score(y_test, y_prob_scaled)

cm_scaled = confusion_matrix(y_test, y_pred_scaled)

print('2.Baseline Logistic Regression with StandardScaler')
print("Accuracy:", accuracy_scaled)
print("Precision:", precision_scaled)
print("Recall:", recall_scaled)
print("F1-score:", f1_scaled)
print("ROC-AUC:", roc_auc_scaled)

print("\nConfusion Matrix:")
print(cm_scaled)

2.Baseline Logistic Regression with StandardScaler
Accuracy: 0.9974415464430388
Precision: 0.6857142857142857
Recall: 0.4897959183673469
F1-score: 0.5714285714285714
ROC-AUC: 0.9928963669335942

Confusion Matrix:
[[28022    22]
 [   50    48]]


### 3. Balanced Logistic Regression with scaling

In [24]:
model_balanced = LogisticRegression(max_iter=1000,class_weight='balanced')
model_balanced.fit(X_train_scaled, y_train)

y_pred_balanced = model_balanced.predict(X_test_scaled)
y_prob_balanced = model_balanced.predict_proba(X_test_scaled)[:,1]

accuracy_balanced = accuracy_score(y_test, y_pred_balanced)
precision_balanced = precision_score(y_test, y_pred_balanced)
recall_balanced = recall_score(y_test, y_pred_balanced)
f1_balanced = f1_score(y_test, y_pred_balanced)
roc_auc_balanced = roc_auc_score(y_test, y_prob_balanced)

cm_balanced = confusion_matrix(y_test, y_pred_balanced)

print('3. Balanced Logistic Regression with scaling')
print("Accuracy:", accuracy_balanced)
print("Precision:", precision_balanced)
print("Recall:", recall_balanced)
print("F1-score:", f1_balanced)
print("ROC-AUC:", roc_auc_balanced)

print("\nConfusion Matrix:")
print(cm_balanced)

3. Balanced Logistic Regression with scaling
Accuracy: 0.9774003269135101
Precision: 0.12742382271468145
Recall: 0.9387755102040817
F1-score: 0.22439024390243903
ROC-AUC: 0.9947968061850329

Confusion Matrix:
[[27414   630]
 [    6    92]]


### 4. Logistic Regression with Stratified K-Fold using pipeline and scaling

In [25]:
pipeline = Pipeline([("scaler", StandardScaler()),("model", LogisticRegression(max_iter=1000))])

cv = StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

cv_results = cross_validate( pipeline,X,y, cv=cv,scoring=scoring)

print('Logistic Regression with Stratified K-Fold using pipeline and scaling')
print("Accuracy:", cv_results["test_accuracy"].mean())
print("Precision:", cv_results["test_precision"].mean())
print("Recall:", cv_results["test_recall"].mean())
print("F1-score:", cv_results["test_f1"].mean())
print("ROC-AUC:", cv_results["test_roc_auc"].mean())

Logistic Regression with Stratified K-Fold using pipeline and scaling
Accuracy: 0.9977328748622083
Precision: 0.7519472766159055
Recall: 0.5183673469387755
F1-score: 0.6131246626359032
ROC-AUC: 0.9813919826717568


### 5. Logistic Regression with Grid Search using pipeline and scaling 

In [26]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000
    ))
])

param_dist = {
    "model__C": [0.01, 0.1, 1, 10],
    "model__penalty": ["l1", "l2"],
    "model__solver": ["liblinear"]
}

search = RandomizedSearchCV(
    pipeline,
    param_distributions=param_dist,
    n_iter=5,          
    cv=3,              
    scoring="f1_weighted",
    n_jobs=-1,
    verbose=2,
    random_state=42
)

search.fit(X, y)

best_model = search.best_estimator_

print("Best parameters:", search.best_params_)

y_pred = best_model.predict(X)
y_prob = best_model.predict_proba(X)[:, 1]

accuracy = accuracy_score(y, y_pred)
precision = precision_score(y, y_pred)
recall = recall_score(y, y_pred)
f1 = f1_score(y, y_pred)
roc_auc = roc_auc_score(y, y_prob)

cm = confusion_matrix(y, y_pred)

print("Random Search")
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)
print("ROC-AUC:", roc_auc)

print("\nConfusion Matrix:")
print(cm)


Fitting 3 folds for each of 5 candidates, totalling 15 fits
Best parameters: {'model__solver': 'liblinear', 'model__penalty': 'l2', 'model__C': 1}
Random Search
Accuracy: 0.9978252681103286
Precision: 0.782608695652174
Recall: 0.5163934426229508
F1-score: 0.6222222222222222
ROC-AUC: 0.9842476359287621

Confusion Matrix:
[[140149     70]
 [   236    252]]


In [5]:
import pandas as pd

results_1500 = [
    {"Model": "LR baseline", "Accuracy": 0.99733, "Precision": 0.6825, "Recall": 0.4388, "F1": 0.5342, "ROC-AUC": 0.9616},
    {"Model": "LR + scaling", "Accuracy": 0.99744, "Precision": 0.6857, "Recall": 0.4898, "F1": 0.5714, "ROC-AUC": 0.9929},
    {"Model": "LR balanced", "Accuracy": 0.97740, "Precision": 0.1274, "Recall": 0.9388, "F1": 0.2244, "ROC-AUC": 0.9948},
    {"Model": "LR + Stratified CV", "Accuracy": 0.99773, "Precision": 0.7519, "Recall": 0.5184, "F1": 0.6131, "ROC-AUC": 0.9814},
    {"Model": "LR tuned", "Accuracy": 0.99783, "Precision": 0.7826, "Recall": 0.5164, "F1": 0.6222, "ROC-AUC": 0.9842}
]

df_1500 = pd.DataFrame(results_1500)
df_1500.sort_values(by="F1", ascending=False).round(4)

,Model,Accuracy,Precision,Recall,F1,ROC-AUC
4,LR tuned,0.9978,0.7826,0.5164,0.6222,0.9842
3,LR + Stratified CV,0.9977,0.7519,0.5184,0.6131,0.9814
1,LR + scaling,0.9974,0.6857,0.4898,0.5714,0.9929
0,LR baseline,0.9973,0.6825,0.4388,0.5342,0.9616
2,LR balanced,0.9774,0.1274,0.9388,0.2244,0.9948


Základný model bez škálovania vykazoval hodnotu F1 ~0,53, čo je podobné predchádzajúcim výsledkom. Najlepšie výsledky dosiahol model logistickej regresie so škálovaním a výberom hyperparametrov. Najhoršie výsledky opäť dosiahol model s vyvažovaním tried